In [7]:
# Pre-training SoRL on arithmatic generalization dataset
# ------------------------------------------------------ 
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

assert BOS_TOKEN_ID == 20, "BOS_TOKEN_ID must be set to 20 (go to 'sorl/gat_sim.py' to change it)"
gat_config = GATConfig(
    vocab_sizes=[20, 6],  # 6 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

Generate multiplication data: 

```python data/arithmetic.py```

In [2]:
# ---- Arithmatic generalization dataset loader ----
from sorl.arithmetic import data_generator
from sorl.arithmetic import DigitTokenizer

# --- tokenizer ---
tokenizer = DigitTokenizer()

# --- data loader ---
train_loader = data_generator(filename_pattern="data/multiplication/multiplication_train.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/multiplication/multiplication_val_id.bin", sequence_length=64, device="cpu")

In [ ]:
from sorl.neo_utils import sorl_search, sorl_search_v2, compute_loss, sorl_evaluate, compute_vocab_utilization_rate
from sorl.gapt import GatedPhaseTransition

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

# memory_span = 1
memory_span = 25
attn_blocksize = 1792
K = 4
n = 4  # number of rollout
max_iterations = 2
temperature = 10.0
alpha = 0.0
num_steps = 500
gapt = GatedPhaseTransition(p_m=10)

for step in range(num_steps): 
 
    optimizer.zero_grad()

    tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv, pt_curiosity = sorl_search_v2(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature,
                                                               truncate_seq_len=False)
    
    # --- compute loss ---
    traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize,
                                       per_pos_curiosity=pt_curiosity)
                                       
    loss = gapt.step(traj_loss, abs_loss, verbose=True)
    # loss = traj_loss + abs_loss

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(search_tokens, model, n=4, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=10.0,
                                                                   truncate_seq_len=False)
            traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize)
            vocab_util_rate = compute_vocab_utilization_rate(val_tokens, model)
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}% | curiosity std: {pt_curiosity.std().item():.2f} | vocab utilization rate: {vocab_util_rate[0] * 100:.2f}%")

validation step 0 | traj_loss: 3.21 | abs_loss: 3.28 | search adv: -0.06% | curiosity std: 0.00
validation step 2 | traj_loss: 3.09 | abs_loss: 3.40 | search adv: -0.39% | curiosity std: 0.00
validation step 4 | traj_loss: 2.94 | abs_loss: 3.64 | search adv: -0.33% | curiosity std: 0.00
validation step 6 | traj_loss: 2.82 | abs_loss: 3.87 | search adv: -0.12% | curiosity std: 0.00
validation step 8 | traj_loss: 2.73 | abs_loss: 4.10 | search adv: 0.06% | curiosity std: 0.00
validation step 10 | traj_loss: 2.62 | abs_loss: 4.32 | search adv: -0.01% | curiosity std: 0.00
validation step 12 | traj_loss: 2.50 | abs_loss: 4.45 | search adv: -0.06% | curiosity std: 0.00
validation step 14 | traj_loss: 2.48 | abs_loss: 4.54 | search adv: -0.11% | curiosity std: 0.00
validation step 16 | traj_loss: 2.35 | abs_loss: 4.69 | search adv: -0.11% | curiosity std: 0.00
validation step 18 | traj_loss: 2.30 | abs_loss: 4.79 | search adv: 0.03% | curiosity std: 0.00
validation step 20 | traj_loss: 2.29 

In [6]:
# generate function implementation 
# ----------------------------------
from sorl.neo_utils import generate
from sorl.arithmetic import process_query, check_answer

K = 5
tokens = next(val_loader)
idx, answer_idx = process_query(tokens)

print(f"init   | idx: {idx[0].tolist()} | question: {tokenizer.decode(idx[0].tolist()[1:-1])}")
for i in range(len(answer_idx[0])*2): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=0.0)
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    print(f"                         idx : {idx[0].tolist()}")

is_correct, pred_answer, true_answer = check_answer(idx_without_abstraction, answer_idx, tokenizer)
print(f"is_correct: {is_correct} | pred_answer: {pred_answer} | true_answer: {true_answer}")

init   | idx: [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3] | question: 46 x 268 
step 1 idx (abstraction free): [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3, 4]
                         idx : [0, 14, 16, 4, 2, 22, 4, 12, 16, 18, 4, 25, 3, 4]
step 2 idx (abstraction free): [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3, 4, 11]
                         idx : [0, 14, 16, 4, 2, 22, 4, 12, 16, 18, 4, 25, 3, 4, 11]
step 3 idx (abstraction free): [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3, 4, 11, 12]
                         idx : [0, 14, 16, 4, 2, 22, 4, 12, 16, 18, 4, 25, 3, 4, 11, 12]
step 4 idx (abstraction free): [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3, 4, 11, 12, 12]
                         idx : [0, 14, 16, 4, 2, 22, 4, 12, 16, 18, 4, 25, 3, 4, 11, 12, 12]
step 5 idx (abstraction free): [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3, 4, 11, 12, 12, 18]
                         idx : [0, 14, 16, 4, 2, 22, 4, 12, 16, 18, 4, 25, 3, 4, 11, 12, 12, 25, 18]
step 6 idx (abstraction free): [0, 14, 16, 4, 2, 4, 12, 16, 18, 4, 3, 4